# Tutorial 04 — End-to-End Scenario F6: `1.0.0 -> 2.0.0` Transitions, Agreements & Revocation

This notebook reproduces step by step via HTTP requests the **Normative Scenario F6** from the *Vocabulary Hub v0.2.1* specification, traveling across more than two months (`2026-09-01` to `2026-11-12`) using the shared injectable clock (`clock_file`, `DP-9`).

### Scenario F6 Timeline
1. **`2026-09-01T10:00:00Z`:** Version **`1.0.0`** (`initial`) is published. Permits `m1 (odrl:use)` and `m2 (ex:access)`. Rejects `m3 (odrl:read)`.
2. **`2026-10-01T10:00:00Z`:** Version **`2.0.0`** (`breaking`, with `propagation_window = P2D` and `grace_window = P30D`) is published:
   - `new_content_from = 2026-10-03T10:00:00Z`
   - `enforced_from = 2026-11-02T10:00:00Z`
   - Retires `ex:access` (after deprecation in `1.1.0`) and adopts `odrl:read`.
3. **`2026-10-02T12:00:00Z` (Propagation Window):** `2.0.0` is `Announced`. Only `1.0.0` is in force.
4. **`2026-10-03T09:59:55Z` and `2026-10-10T12:00:00Z` (Grace Window / `Transition`):** Both versions are in $V_t$. `m2 (ex:access)` is accepted with `VHub-Verdict: warning;...;code="VERDICT_NOT_IN_HEAD";version="1.0.0";sunset=@1793613600`.
5. **`2026-10-10T12:00:00Z` (Agreement `A1`):** Agreement `urn:uuid:A1` is signed using `ex:access` (version `1.0.0`) through the `dsp-egress` forward proxy.
6. **`2026-11-02T10:00:05Z` (`lease_margin` `L = 10s`):** 5 seconds after `enforced_from`, `m2` is still accepted under the `L` margin (`REQ-SYS-007`).
7. **`2026-11-05T12:00:00Z` (`1.0.0` Retired):** New offers with `m2 (ex:access)` are rejected (`400 VERDICT_VIOLATION`), but **transfers under agreement `A1` remain permitted (`200 OK`)** because `1.0.0` was only retired, not revoked (`REQ-CLI-016`, `REQ-CLI-017`).
8. **`2026-11-10T00:00:00Z` -> `2026-11-12T00:00:10Z` (`1.0.0` Revocation):** The authority revokes `1.0.0` (`revoked_from = 2026-11-12T00:00:00Z`). The client emits `AGREEMENT_REVOCATION_SCHEDULED` and, once `revoked_from + L` (`2026-11-12T00:00:10Z`) is reached, blocks `A1` transfers with `400 AGREEMENT_VERSION_REVOKED`.

In [1]:
from lab import AlexandriaLab, http_request, show_events_table, show_f6_matrix

lab = AlexandriaLab(compiler="dsl", initial_clock="2026-09-01T10:00:00Z")
lab.start_all()

PROFILE = "https://example.com/odrl/profile/f6"

def contract_req(action: str) -> dict:
    return {
        "@context": ["https://w3id.org/dspace/2025/1/context.jsonld", f"{PROFILE}/context.jsonld"],
        "@type": "ContractRequestMessage",
        "consumerPid": "urn:uuid:c-1",
        "offer": {
            "@id": "urn:uuid:offer-1",
            "@type": "Offer",
            "profile": PROFILE,
            "target": "urn:uuid:ds-1",
            "permission": [{"action": action}],
        },
        "callbackAddress": "http://consumer/cb",
    }

def agreement_msg() -> dict:
    return {
        "@context": ["https://w3id.org/dspace/2025/1/context.jsonld"],
        "@type": "ContractAgreementMessage",
        "providerPid": "urn:uuid:p-1",
        "consumerPid": "urn:uuid:c-1",
        "agreement": {
            "@id": "urn:uuid:A1",
            "profile": f"{PROFILE}/1.0.0",
            "@type": "Agreement",
            "target": "urn:uuid:ds-1",
            "permission": [{"action": "ex:access"}],
        },
        "callbackAddress": "http://provider/cb",
    }

def transfer_msg() -> dict:
    return {
        "@context": ["https://w3id.org/dspace/2025/1/context.jsonld"],
        "@type": "TransferRequestMessage",
        "consumerPid": "urn:uuid:t-1",
        "agreementId": "urn:uuid:A1",
        "format": "HTTP_PULL",
        "callbackAddress": "http://consumer/cb",
    }

def publish_bundle(version_str: str) -> None:
    yaml_body = (lab.root / f"conformance/f6/drafts/v{version_str}.yaml").read_text(encoding="utf-8")
    _, _, d = http_request("POST", f"{lab.hub_url}/v1/drafts", token="editor-token", body=yaml_body, verbose=False)
    _, _, c = http_request("POST", f"{lab.hub_url}/v1/drafts/{d['id']}:propose", token="editor-token", verbose=False)
    st, _, rec = http_request("POST", f"{lab.hub_url}/v1/candidates/{c['id']}:approve", token="releaser-token", body={}, verbose=False)
    assert st == 201
    r = rec["signed"]["record"]
    print(f"📦 Published version {r['version']} ({r['class']}): published={r['published_at']} new_content={r['new_content_from']} enforced={r['enforced_from']}")

✔ alexandria-server listening at http://127.0.0.1:62658 (DID: did:web:127.0.0.1%3A62658)

✔ alexandria-client active | dsp-ingress: http://127.0.0.1:62660 | dsp-egress: http://127.0.0.1:62662 | admin: http://127.0.0.1:62663

## Phase 1: Publishing `1.0.0` (`2026-09-01`) and `2.0.0` (`2026-10-01`)

In [2]:
lab.set_clock("2026-09-01T10:00:00Z")
publish_bundle("1.0.0")

lab.set_clock("2026-10-01T10:00:00Z")
publish_bundle("2.0.0")

print("\nCurrent version listing on the Hub (GET /v1/versions):")
http_request("GET", f"{lab.hub_url}/v1/versions", token="member-token")

📦 Published version 1.0.0 (initial): published=2026-09-01T10:00:00Z new_content=2026-09-03T10:00:00Z enforced=2026-09-03T10:00:00Z
📦 Published version 2.0.0 (breaking): published=2026-10-01T10:00:00Z new_content=2026-10-03T10:00:00Z enforced=2026-11-02T10:00:00Z

Current version listing on the Hub (GET /v1/versions):


🕒 Clock updated → 2026-09-01T10:00:00Z

🕒 Clock updated → 2026-10-01T10:00:00Z

cache-control,"private, no-cache"
connection,close
content-length,876
content-type,application/json
date,"Thu, 01 Oct 2026 10:00:00 GMT"


## Phase 2: Propagation Window (`2026-10-02T12:00:00Z`)

At `2026-10-02T12:00:00Z`, version `2.0.0` is published but its `new_content_from` is `2026-10-03T10:00:00Z` (still `Announced`).
Therefore, only `1.0.0` is in the effective set $V_t$:
- `m1 (odrl:use)` -> `200 OK`
- `m2 (ex:access)` -> `200 OK`
- `m3 (odrl:read)` -> `400 Bad Request` (`VERDICT_VIOLATION`)

In [3]:
lab.set_clock("2026-10-02T12:00:00Z")
lab.wait_synced("2026-10-02T12:00:00Z")

for label, act in [("m1 odrl:use", "odrl:use"), ("m2 ex:access", "ex:access"), ("m3 odrl:read", "odrl:read")]:
    print(f"--- {label} @ 2026-10-02T12:00:00Z ---")
    http_request("POST", f"{lab.dsp_ingress_url}/negotiations/request", body=contract_req(act))

--- m1 odrl:use @ 2026-10-02T12:00:00Z ---
--- m2 ex:access @ 2026-10-02T12:00:00Z ---
--- m3 odrl:read @ 2026-10-02T12:00:00Z ---


🕒 Clock updated → 2026-10-02T12:00:00Z

connection,close
content-length,55
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:46 GMT"


connection,close
content-length,55
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:46 GMT"


connection,close
content-length,184
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:46 GMT"
vhub-verdict,"violation;gate=policy_intake;code=""VERDICT_VIOLATION"""


## Phase 3: Grace Window / Transition (`2026-10-03T09:59:55Z` and `2026-10-10T12:00:00Z`)

Thanks to the `lease_margin` (`L = 10s`), `2.0.0` enters $V_t$ at `new_content_from - L` (`2026-10-03T09:59:50Z`).
Now `2.0.0` is the head version and `1.0.0` is in **`Transition`** (in grace until `2026-11-02T10:00:00Z` = `@1793613600`):
- `m1 (odrl:use)` and `m3 (odrl:read)` pass directly against `2.0.0` (`200 OK` without warnings).
- `m2 (ex:access)` fails against `2.0.0` but passes against `1.0.0`: it is forwarded (`200 OK`, `forward-notice`) with the header:
  `VHub-Verdict: warning;gate=policy_intake;code="VERDICT_NOT_IN_HEAD";version="1.0.0";sunset=@1793613600`

In [4]:
matrix_rows = []
for instant in ["2026-10-03T09:59:55Z", "2026-10-10T12:00:00Z"]:
    lab.set_clock(instant)
    lab.wait_synced(instant)
    for label, act in [("m1 odrl:use", "odrl:use"), ("m2 ex:access", "ex:access"), ("m3 odrl:read", "odrl:read")]:
        st, hdr, _ = http_request("POST", f"{lab.dsp_ingress_url}/negotiations/request", body=contract_req(act), verbose=False)
        matrix_rows.append({"instant": instant, "phase": "Transition (Grace)", "message": label, "status": str(st), "verdict": hdr.get("vhub-verdict", "OK (2.0.0 head)")})
show_f6_matrix(matrix_rows, title="Evaluation During the Grace Window (2.0.0 Head + 1.0.0 in Transition)")

🕒 Clock updated → 2026-10-03T09:59:55Z

🕒 Clock updated → 2026-10-10T12:00:00Z

Instant (clock_file),Lifecycle Phase,Message / Action,HTTP,VHub-Verdict Header / Code
2026-10-03T09:59:55Z,Transition (Grace),m1 odrl:use,HTTP 200,OK (2.0.0 head)
2026-10-03T09:59:55Z,Transition (Grace),m2 ex:access,HTTP 200,"warning;gate=policy_intake;code=""VERDICT_NOT_IN_HEAD"";version=""1.0.0"";sunset=@1793613600"
2026-10-03T09:59:55Z,Transition (Grace),m3 odrl:read,HTTP 200,OK (2.0.0 head)
2026-10-10T12:00:00Z,Transition (Grace),m1 odrl:use,HTTP 200,OK (2.0.0 head)
2026-10-10T12:00:00Z,Transition (Grace),m2 ex:access,HTTP 200,"warning;gate=policy_intake;code=""VERDICT_NOT_IN_HEAD"";version=""1.0.0"";sunset=@1793613600"
2026-10-10T12:00:00Z,Transition (Grace),m3 odrl:read,HTTP 200,OK (2.0.0 head)


## Phase 4: Capturing Agreement `A1` via the `dsp-egress` Forward Proxy (`2026-10-10T12:00:00Z`)

While `1.0.0` is still in its grace window, the provider sends the `ContractAgreementMessage` for agreement `urn:uuid:A1` (using `ex:access` under `1.0.0`) through the `dsp-egress` forward proxy.
`alexandria-client` validates it (`warning` / `forward-notice`) and, upon receiving `200 OK` from the connector, **automatically records agreement `urn:uuid:A1` bound to version `1.0.0`** (`REQ-CLI-015`).

In [5]:
print("1) Send ContractAgreementMessage (urn:uuid:A1) through the dsp-egress proxy:")
st_ag, hdr_ag, _ = http_request(
    "POST",
    f"{lab.conn_url}/negotiations/c-1/agreement",
    body=agreement_msg(),
    proxy=lab.dsp_egress_url,
)
assert st_ag == 200
assert "VERDICT_NOT_IN_HEAD" in hdr_ag.get("vhub-verdict", "")

print("2) Query the recorded agreement on the Admin API (GET /agreements/urn:uuid:A1):")
st_get_ag, _, ag_doc = http_request("GET", f"{lab.admin_url}/agreements/urn:uuid:A1")
assert st_get_ag == 200
assert ag_doc["version"] == "1.0.0"

1) Send ContractAgreementMessage (urn:uuid:A1) through the dsp-egress proxy:
2) Query the recorded agreement on the Admin API (GET /agreements/urn:uuid:A1):


connection,close
content-length,61
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:48 GMT"
vhub-verdict,"warning;gate=policy_intake;code=""VERDICT_NOT_IN_HEAD"";version=""1.0.0"";sunset=@1793613600"


connection,close
content-length,333
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:48 GMT"


## Phase 5: `lease_margin` (`2026-11-02T10:00:05Z`) and Retirement of `1.0.0` (`2026-11-05T12:00:00Z`)

- At **`2026-11-02T10:00:05Z`** (5 s after `enforced_from`, still within `L = 10s`), `m2 ex:access` is still accepted with a warning (`REQ-SYS-007`).
- At **`2026-11-05T12:00:00Z`** (past `enforced_from + L`), `1.0.0` transitions to **`Retired`**:
  - A new negotiation request with `m2 (ex:access)` is **rejected (`400 VERDICT_VIOLATION`)**.
  - Conversely, a transfer request (`POST /transfers/request`) under the **already-signed agreement `urn:uuid:A1`** **is accepted (`200 OK`)** because `alexandria-client` retains package `1.0.0` for its active agreements (`REQ-CLI-016`, `REQ-CLI-017`).

In [6]:
# Synchronize listing at 2026-11-02T06:00:00Z and test at 2026-11-02T10:00:05Z (within enforced_from + 10s)
lab.set_clock("2026-11-02T06:00:00Z")
lab.wait_synced("2026-11-02T06:00:00Z")
lab.set_clock("2026-11-02T10:00:05Z")

print("1) At 2026-11-02T10:00:05Z (within enforced_from + L=10s), m2 still passes with warning:")
st_l, hdr_l, _ = http_request("POST", f"{lab.dsp_ingress_url}/negotiations/request", body=contract_req("ex:access"))
assert st_l == 200
assert "VERDICT_NOT_IN_HEAD" in hdr_l.get("vhub-verdict", "")

# Advance clock to 2026-11-05T12:00:00Z (1.0.0 is now Retired)
lab.set_clock("2026-11-05T12:00:00Z")
lab.wait_synced("2026-11-05T12:00:00Z")

print("2) At 2026-11-05T12:00:00Z (1.0.0 Retired), a new offer with m2 ex:access is REJECTED (400):")
st_r, _, _ = http_request("POST", f"{lab.dsp_ingress_url}/negotiations/request", body=contract_req("ex:access"))
assert st_r == 400

print("3) However, a transfer under the already-signed agreement A1 PASSES (200 OK):")
st_tr1, _, _ = http_request("POST", f"{lab.dsp_ingress_url}/transfers/request", body=transfer_msg())
assert st_tr1 == 200

1) At 2026-11-02T10:00:05Z (within enforced_from + L=10s), m2 still passes with warning:
2) At 2026-11-05T12:00:00Z (1.0.0 Retired), a new offer with m2 ex:access is REJECTED (400):
3) However, a transfer under the already-signed agreement A1 PASSES (200 OK):


🕒 Clock updated → 2026-11-02T06:00:00Z

🕒 Clock updated → 2026-11-02T10:00:05Z

connection,close
content-length,55
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:49 GMT"
vhub-verdict,"warning;gate=policy_intake;code=""VERDICT_NOT_IN_HEAD"";version=""1.0.0"";sunset=@1793613600"


🕒 Clock updated → 2026-11-05T12:00:00Z

connection,close
content-length,184
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:50 GMT"
vhub-verdict,"violation;gate=policy_intake;code=""VERDICT_VIOLATION"""


connection,close
content-length,52
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:50 GMT"


## Phase 6: Revocation of `1.0.0` (`2026-11-10` -> `2026-11-12`)

1. At **`2026-11-10T00:00:00Z`**, the authority revokes version `1.0.0` (`POST /v1/versions/1.0.0:revoke`). With `propagation_window = P2D`, `revoked_from` is set to `2026-11-12T00:00:00Z`.
2. Upon syncing the updated listing, `alexandria-client` detects that agreement `urn:uuid:A1` depends on `1.0.0` and emits **`AGREEMENT_REVOCATION_SCHEDULED`**.
3. At **`2026-11-12T00:00:05Z`** (within `revoked_from + L`), the `A1` transfer still passes (`200 OK`).
4. At **`2026-11-12T00:00:10Z`** (once `revoked_from + L` is reached), the `A1` transfer is **blocked (`400 Bad Request`)** with `AGREEMENT_VERSION_REVOKED`.

In [7]:
lab.set_clock("2026-11-10T00:00:00Z")
print("1) Authority revokes version 1.0.0 (POST /v1/versions/1.0.0:revoke):")
st_rev, _, _ = http_request(
    "POST",
    f"{lab.hub_url}/v1/versions/1.0.0:revoke",
    token="releaser-token",
    body={"justification": "superseded by 2.0.0"},
)
assert st_rev == 200
lab.wait_synced("2026-11-10T00:00:00Z")

# Check the AGREEMENT_REVOCATION_SCHEDULED event (type vhub.agreement)
rev_events = [e for e in lab.read_events() if e.get("code") == "AGREEMENT_REVOCATION_SCHEDULED" or "AGREEMENT_REVOCATION_SCHEDULED" in e.get("codes", [])]
show_events_table(rev_events, title="Scheduled Revocation Notice Emitted by Client")

# At 2026-11-12T00:00:05Z (within L=10s after revoked_from)
lab.set_clock("2026-11-12T00:00:05Z")
lab.wait_synced("2026-11-12T00:00:05Z")
print("\n2) Transfer of A1 at 2026-11-12T00:00:05Z (within revoked_from + L -> expected 200 OK):")
st_tr2, _, _ = http_request("POST", f"{lab.dsp_ingress_url}/transfers/request", body=transfer_msg())
assert st_tr2 == 200

# At 2026-11-12T00:00:10Z (reached revoked_from + L)
lab.set_clock("2026-11-12T00:00:10Z")
print("3) Transfer of A1 at 2026-11-12T00:00:10Z (reached revoked_from + L -> expected 400 AGREEMENT_VERSION_REVOKED):")
st_tr3, hdr_tr3, body_tr3 = http_request("POST", f"{lab.dsp_ingress_url}/transfers/request", body=transfer_msg())
assert st_tr3 == 400
assert body_tr3["code"] == "AGREEMENT_VERSION_REVOKED"

lab.close()

1) Authority revokes version 1.0.0 (POST /v1/versions/1.0.0:revoke):

2) Transfer of A1 at 2026-11-12T00:00:05Z (within revoked_from + L -> expected 200 OK):
3) Transfer of A1 at 2026-11-12T00:00:10Z (reached revoked_from + L -> expected 400 AGREEMENT_VERSION_REVOKED):


🕒 Clock updated → 2026-11-10T00:00:00Z

connection,close
content-length,573
content-type,application/json
date,"Tue, 10 Nov 2026 00:00:00 GMT"


Type,Instant (UTC),Mode / Sev,Route / Version,Classification / Agreement,Codes
vhub.agreement,2026-11-10T00:00:00Z,info,1.0.0,urn:uuid:A1,AGREEMENT_REVOCATION_SCHEDULED


🕒 Clock updated → 2026-11-12T00:00:05Z

connection,close
content-length,52
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:52 GMT"


🕒 Clock updated → 2026-11-12T00:00:10Z

connection,close
content-length,189
content-type,application/json
date,"Tue, 10 Nov 2026 06:37:52 GMT"
vhub-verdict,"violation;gate=agreement_admission;code=""AGREEMENT_VERSION_REVOKED"""
